In [1]:
import os, random, time, json
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score
from imblearn.metrics import specificity_score
from mambapy.vim import VMamba, MambaConfig
from thop import profile

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


In [2]:
AUG_ROOT = "D:/mamba_model/aug_clean_tio"       
TAG      = "tio"                                 
# ───────────────────────────────────────────────────────────────

COHORT_CSV = "D:/mamba_model/thesis_cohort_clean.csv"
MRI_CACHE  = f"{AUG_ROOT}/roi_mri"
PET_CACHE  = f"{AUG_ROOT}/roi_pet"
CKPT_DIR   = f"D:/mamba_model/checkpoints_v7_roi_{TAG}"
RESULTS    = f"D:/mamba_model/v7_roi_{TAG}_results.json"
os.makedirs(CKPT_DIR, exist_ok=True)

SPLIT_SEED  = 42
AUG_SEEDS   = [1, 101, 42]
BATCH_SIZE  = 4
NUM_WORKERS = 0          

print(f"aug:  {AUG_ROOT}")
print(f"ckpt: {CKPT_DIR}")
for p in (MRI_CACHE, PET_CACHE):
    n = len(os.listdir(p)) if os.path.isdir(p) else 0
    print(f"  {os.path.basename(p)}: {n} files{'  *** MISSING ***' if n == 0 else ''}")

aug:  D:/mamba_model/aug_clean_tio
ckpt: D:/mamba_model/checkpoints_v7_roi_tio
  roi_mri: 560 files
  roi_pet: 0 files  *** MISSING ***


In [3]:
class VimEncoder(nn.Module):
    """Bidirectional Mamba over a token sequence. No CNN, no pretraining."""
    def __init__(self, d_model=32, n_layers=2, d_state=16):
        super().__init__()
        cfg = MambaConfig(d_model=d_model, n_layers=n_layers, d_state=d_state,
                          bidirectional=True, divide_output=True,
                          pscan=True, use_cuda=False)
        self.encoder = VMamba(cfg)
        self.final_norm = nn.LayerNorm(d_model)
    def forward(self, tokens):
        return self.final_norm(self.encoder(tokens))

In [4]:
class ROIPatchEmbed3D(nn.Module):
    """6 ROIs -> non-overlapping 8^3 patches -> one token each. 3072 tokens."""
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=32):
        super().__init__()
        self.n_rois, self.patch_size = n_rois, patch_size
        self.grid_size = roi_size // patch_size
        self.patches_per_roi = self.grid_size ** 3
        self.d_model = d_model
        self.patch_conv = nn.Conv3d(1, d_model, kernel_size=patch_size, stride=patch_size)
        self.roi_embed    = nn.Embedding(n_rois, d_model)
        self.depth_embed  = nn.Embedding(self.grid_size, d_model)
        self.height_embed = nn.Embedding(self.grid_size, d_model)
        self.width_embed  = nn.Embedding(self.grid_size, d_model)
        with torch.no_grad():
            for e in [self.roi_embed, self.depth_embed, self.height_embed, self.width_embed]:
                e.weight.mul_(0.02)
        d, h, w = torch.meshgrid(torch.arange(self.grid_size), torch.arange(self.grid_size),
                                 torch.arange(self.grid_size), indexing="ij")
        self.register_buffer("coordinates", torch.stack([d, h, w], -1).reshape(-1, 3),
                             persistent=False)

    def forward(self, rois):
        B, n = rois.shape[:2]
        x = rois.reshape(B * n, 1, *rois.shape[-3:])
        tokens = self.patch_conv(x).flatten(2).transpose(1, 2)
        tokens = tokens.reshape(B, n, self.patches_per_roi, self.d_model)
        c = self.coordinates
        spatial = (self.depth_embed(c[:, 0]) + self.height_embed(c[:, 1])
                   + self.width_embed(c[:, 2]))
        tokens = tokens + spatial[None, None] + self.roi_embed.weight[None, :, None, :]
        occ = F.max_pool3d((x.abs() > 1e-6).float(),
                           kernel_size=self.patch_size, stride=self.patch_size)
        valid = occ.flatten(1).bool().reshape(B, n, self.patches_per_roi)
        tokens = tokens.reshape(B, -1, self.d_model)
        valid = valid.reshape(B, -1)
        return tokens * valid.unsqueeze(-1).to(tokens.dtype), valid


class VisionMambaBranch(nn.Module):
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=32,
                 n_layers=2, d_state=16):
        super().__init__()
        self.n_rois = n_rois
        self.patch_embed = ROIPatchEmbed3D(n_rois, roi_size, patch_size, d_model)
        self.vim = VimEncoder(d_model, n_layers, d_state)
    def forward(self, rois):
        tokens, valid = self.patch_embed(rois)
        tokens = self.vim(tokens)
        w = valid.unsqueeze(-1).to(tokens.dtype)
        return (tokens * w).sum(1) / w.sum(1).clamp_min(1.0)


class VisionMambaModel(nn.Module):
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=32,
                 n_layers=2, n_classes=2, d_state=16, dropout=0.4):
        super().__init__()
        self.branch = VisionMambaBranch(n_rois, roi_size, patch_size, d_model, n_layers, d_state)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, n_classes)
    def forward(self, rois):
        return self.classifier(self.dropout(self.branch(rois)))


class MultimodalVisionMambaModel(nn.Module):
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=32,
                 n_layers=2, n_classes=2, d_state=16, dropout=0.4):
        super().__init__()
        self.mri_branch = VisionMambaBranch(n_rois, roi_size, patch_size, d_model, n_layers, d_state)
        self.pet_branch = VisionMambaBranch(n_rois, roi_size, patch_size, d_model, n_layers, d_state)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model * 2, n_classes)
    def forward(self, mri, pet):
        f = torch.cat([self.mri_branch(mri), self.pet_branch(pet)], dim=1)
        return self.classifier(self.dropout(f))

In [5]:
df = pd.read_csv(COHORT_CSV)
sessions, labels = df["mri_session"].values, df["outcome_label"].values

X_tv, X_test, y_tv, y_test = train_test_split(
    sessions, labels, test_size=0.2, random_state=SPLIT_SEED, stratify=labels)
X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv, test_size=0.25, random_state=SPLIT_SEED, stratify=y_tv)

session_to_subject = dict(zip(df["mri_session"], df["subject_id"]))
print(f"train {len(X_train)} | val {len(X_val)} | test {len(X_test)} "
      f"| test pos {int(y_test.sum())}")


# ── in-memory cache: first epoch reads from disk, the rest from RAM ──
# ~3.4 GB per modality (560 files x 6 MB)
_CACHE = {}

def load_cached(path):
    a = _CACHE.get(path)
    if a is None:
        a = np.load(path).astype(np.float32)
        _CACHE[path] = a
    return a
# NOTE: torch.from_numpy shares memory with the cached array. Nothing here
# mutates the tensors, but never add an in-place transform downstream.


class ROIDataset(Dataset):
    """Single modality. Training set includes 3 augmented copies per subject."""
    def __init__(self, sessions, labels, cache_dir, is_mri=True, is_train=False):
        self.samples, self.cache_dir = [], cache_dir
        for ses, lab in zip(sessions, labels):
            key = ses if is_mri else session_to_subject[ses]
            self.samples.append((key, lab, "orig"))
            if is_train:
                for s in AUG_SEEDS:
                    self.samples.append((key, lab, f"aug{s}"))
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        key, lab, ver = self.samples[i]
        a = load_cached(f"{self.cache_dir}/{key}_{ver}.npy")
        return torch.from_numpy(a).unsqueeze(1), torch.tensor(lab, dtype=torch.long), key


class MultimodalROIDataset(Dataset):
    """Pairs MRI and PET for the same subject and the same augmentation seed."""
    def __init__(self, sessions, labels, mri_dir, pet_dir, is_train=False):
        self.samples, self.mri_dir, self.pet_dir = [], mri_dir, pet_dir
        for ses, lab in zip(sessions, labels):
            sid = session_to_subject[ses]
            self.samples.append((ses, sid, lab, "orig"))
            if is_train:
                for s in AUG_SEEDS:
                    self.samples.append((ses, sid, lab, f"aug{s}"))
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        mk, pk, lab, ver = self.samples[i]
        m = load_cached(f"{self.mri_dir}/{mk}_{ver}.npy")
        p = load_cached(f"{self.pet_dir}/{pk}_{ver}.npy")
        return (torch.from_numpy(m).unsqueeze(1), torch.from_numpy(p).unsqueeze(1),
                torch.tensor(lab, dtype=torch.long), mk)


def dl(ds, shuffle=False):
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=True,
                      persistent_workers=(NUM_WORKERS > 0))

mri_loaders = (dl(ROIDataset(X_train, y_train, MRI_CACHE, True, True), True),
               dl(ROIDataset(X_val,   y_val,   MRI_CACHE, True, False)),
               dl(ROIDataset(X_test,  y_test,  MRI_CACHE, True, False)))

pet_loaders = (dl(ROIDataset(X_train, y_train, PET_CACHE, False, True), True),
               dl(ROIDataset(X_val,   y_val,   PET_CACHE, False, False)),
               dl(ROIDataset(X_test,  y_test,  PET_CACHE, False, False)))

mm_loaders  = (dl(MultimodalROIDataset(X_train, y_train, MRI_CACHE, PET_CACHE, True), True),
               dl(MultimodalROIDataset(X_val,   y_val,   MRI_CACHE, PET_CACHE, False)),
               dl(MultimodalROIDataset(X_test,  y_test,  MRI_CACHE, PET_CACHE, False)))

print(f"train samples (4x augmented): {len(mri_loaders[0].dataset)}")

# first pass fills the cache from disk, second should be near-instant
t0 = time.time()
for i, _ in enumerate(mri_loaders[0]):
    if i >= 20: break
cold = time.time() - t0

t0 = time.time()
for i, _ in enumerate(mri_loaders[0]):
    if i >= 20: break
warm = time.time() - t0

print(f"20 batches: cold {cold:.1f}s -> warm {warm:.1f}s")
print(f"cached arrays: {len(_CACHE)}  (~{sum(a.nbytes for a in _CACHE.values())/1e9:.1f} GB)")

train 120 | val 40 | test 40 | test pos 20
train samples (4x augmented): 480
20 batches: cold 8.5s -> warm 6.8s
cached arrays: 154  (~1.0 GB)


In [6]:
def train_epoch(model, loader, opt, crit, mm):
    model.train(); tot = 0
    for batch in loader:
        opt.zero_grad()
        if mm:
            a, b, lb, _ = batch; out = model(a.to(device), b.to(device))
        else:
            a, lb, _ = batch;    out = model(a.to(device))
        loss = crit(out, lb.to(device)); loss.backward(); opt.step(); tot += loss.item()
    return tot / len(loader)

def evaluate(model, loader, crit, mm):
    model.eval(); tot, P, L = 0, [], []
    with torch.no_grad():
        for batch in loader:
            if mm:
                a, b, lb, _ = batch; out = model(a.to(device), b.to(device))
            else:
                a, lb, _ = batch;    out = model(a.to(device))
            tot += crit(out, lb.to(device)).item()
            P.extend(out.argmax(1).cpu().numpy()); L.extend(lb.numpy())
    return (tot/len(loader), np.mean(np.array(P) == np.array(L)),
            recall_score(L, P, zero_division=0), specificity_score(L, P))

def measure_inference(model, loader, mm, n=20):
    model.eval(); ts = []
    with torch.no_grad():
        for i, batch in enumerate(loader):
            if i >= n: break
            if mm:
                a, b = batch[0].to(device), batch[1].to(device); bs = a.shape[0]
                if device.type == 'cuda': torch.cuda.synchronize()
                t0 = time.time(); _ = model(a, b)
            else:
                a = batch[0].to(device); bs = a.shape[0]
                if device.type == 'cuda': torch.cuda.synchronize()
                t0 = time.time(); _ = model(a)
            if device.type == 'cuda': torch.cuda.synchronize()
            ts.append((time.time() - t0) / bs)
    return np.mean(ts), np.std(ts)

def compute_flops(model, loader, mm):
    try:
        model.eval(); b = next(iter(loader))
        with torch.no_grad():
            inp = (b[0][:1].to(device), b[1][:1].to(device)) if mm else (b[0][:1].to(device),)
            macs, _ = profile(model, inputs=inp, verbose=False)
        return macs * 2
    except Exception as e:
        print(f"  (FLOPs failed: {e})"); return None


def run_seed(seed, model_cls, loaders, mm, prefix,
             max_epochs=101, patience=15, min_epochs=25, lr=1e-4, log_every=5):
    torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    np.random.seed(seed); random.seed(seed)
    tr, va, te = loaders

    model = model_cls(d_model=32, n_layers=2, n_classes=2, dropout=0.4).to(device)
    crit  = nn.CrossEntropyLoss(label_smoothing=0.05)
    opt   = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    sch   = optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=10)

    best, no_imp, best_ep, total = float('inf'), 0, 0, 0
    path = f"{CKPT_DIR}/{prefix}_seed{seed}.pt"
    print(f"\n--- {prefix} seed {seed} ---")

    for ep in range(1, max_epochs):
        t0 = time.time()
        trl = train_epoch(model, tr, opt, crit, mm)
        vl, vacc, vtpr, vtnr = evaluate(model, va, crit, mm)
        sch.step(vl); dt = time.time() - t0; total += dt
        if ep % log_every == 0 or ep == 1:
            print(f"  ep {ep:>3} | train {trl:.4f} | val {vl:.4f} | "
                  f"acc {vacc:.3f} tpr {vtpr:.3f} tnr {vtnr:.3f} | {dt:.0f}s")
        if vl < best:
            best, best_ep, no_imp = vl, ep, 0
            torch.save(model.state_dict(), path)
        else:
            no_imp += 1
            if ep >= min_epochs and no_imp >= patience:
                print(f"  early stop {ep}, best {best_ep}"); break

    if best_ep < 5:
        print(f"  WARNING: best epoch {best_ep} -- may not have trained")

    model.load_state_dict(torch.load(path, weights_only=True))
    _, acc, tpr, tnr = evaluate(model, te, crit, mm)
    npar = sum(p.numel() for p in model.parameters() if p.requires_grad)
    inf_m, inf_s = measure_inference(model, te, mm)
    fl = compute_flops(model, te, mm)

    print(f"  >>> TEST Acc={acc*100:.1f}% TPR={tpr*100:.1f}% TNR={tnr*100:.1f}% | "
          f"params={npar:,} train={total/60:.1f}min inf={inf_m*1000:.2f}ms "
          f"{f'{fl/1e9:.2f}GFLOPs' if fl else ''} best_ep={best_ep}")

    return {"seed": seed, "acc": acc, "tpr": tpr, "tnr": tnr, "best_epoch": best_ep,
            "train_time_sec": total, "n_params": npar,
            "inf_time_ms": inf_m*1000, "flops": fl}

results = {"mri": [], "pet": [], "mm": []}

In [7]:
results["mri"].append(run_seed(1, VisionMambaModel, mri_loaders, False, "v7_roi_mri"))


--- v7_roi_mri seed 1 ---
  ep   1 | train 0.7003 | val 0.6934 | acc 0.500 tpr 0.900 tnr 0.100 | 45s
  ep   5 | train 0.6921 | val 0.6900 | acc 0.525 tpr 0.900 tnr 0.150 | 7s
  ep  10 | train 0.6854 | val 0.6853 | acc 0.525 tpr 1.000 tnr 0.050 | 8s
  ep  15 | train 0.6786 | val 0.6777 | acc 0.650 tpr 0.800 tnr 0.500 | 8s
  ep  20 | train 0.6641 | val 0.6716 | acc 0.550 tpr 0.850 tnr 0.250 | 7s
  ep  25 | train 0.6411 | val 0.6562 | acc 0.750 tpr 0.600 tnr 0.900 | 7s
  ep  30 | train 0.6079 | val 0.6391 | acc 0.650 tpr 0.600 tnr 0.700 | 7s
  ep  35 | train 0.5568 | val 0.6256 | acc 0.625 tpr 0.700 tnr 0.550 | 8s
  ep  40 | train 0.4764 | val 0.6154 | acc 0.650 tpr 0.600 tnr 0.700 | 8s
  ep  45 | train 0.3983 | val 0.6273 | acc 0.675 tpr 0.600 tnr 0.750 | 7s
  ep  50 | train 0.3131 | val 0.6269 | acc 0.625 tpr 0.600 tnr 0.650 | 7s
  ep  55 | train 0.2529 | val 0.6489 | acc 0.650 tpr 0.700 tnr 0.600 | 7s
  ep  60 | train 0.1908 | val 0.6471 | acc 0.650 tpr 0.650 tnr 0.650 | 7s
  early st

In [8]:
results["mri"].append(run_seed(7, VisionMambaModel, mri_loaders, False, "v7_roi_mri"))


--- v7_roi_mri seed 7 ---
  ep   1 | train 0.7065 | val 0.6930 | acc 0.500 tpr 1.000 tnr 0.000 | 7s
  ep   5 | train 0.6881 | val 0.6874 | acc 0.575 tpr 0.800 tnr 0.350 | 7s
  ep  10 | train 0.6883 | val 0.6835 | acc 0.575 tpr 0.850 tnr 0.300 | 7s
  ep  15 | train 0.6820 | val 0.6790 | acc 0.575 tpr 0.800 tnr 0.350 | 7s
  ep  20 | train 0.6701 | val 0.6717 | acc 0.600 tpr 0.950 tnr 0.250 | 7s
  ep  25 | train 0.6510 | val 0.6595 | acc 0.650 tpr 0.400 tnr 0.900 | 7s
  ep  30 | train 0.6106 | val 0.6444 | acc 0.750 tpr 0.600 tnr 0.900 | 7s
  ep  35 | train 0.5984 | val 0.6531 | acc 0.575 tpr 0.150 tnr 1.000 | 7s
  ep  40 | train 0.5403 | val 0.6283 | acc 0.725 tpr 0.500 tnr 0.950 | 7s
  ep  45 | train 0.4846 | val 0.6158 | acc 0.750 tpr 0.600 tnr 0.900 | 7s
  ep  50 | train 0.4129 | val 0.5964 | acc 0.650 tpr 0.650 tnr 0.650 | 7s
  ep  55 | train 0.3497 | val 0.6106 | acc 0.750 tpr 0.600 tnr 0.900 | 7s
  ep  60 | train 0.2776 | val 0.6862 | acc 0.700 tpr 0.450 tnr 0.950 | 7s
  ep  65 | 

In [9]:
results["mri"].append(run_seed(123, VisionMambaModel, mri_loaders, False, "v7_roi_mri"))


--- v7_roi_mri seed 123 ---
  ep   1 | train 0.6956 | val 0.6921 | acc 0.500 tpr 1.000 tnr 0.000 | 7s
  ep   5 | train 0.6965 | val 0.6900 | acc 0.500 tpr 0.000 tnr 1.000 | 7s
  ep  10 | train 0.6844 | val 0.6796 | acc 0.550 tpr 0.900 tnr 0.200 | 7s
  ep  15 | train 0.6733 | val 0.6724 | acc 0.725 tpr 0.850 tnr 0.600 | 7s
  ep  20 | train 0.6599 | val 0.6720 | acc 0.575 tpr 1.000 tnr 0.150 | 7s
  ep  25 | train 0.6578 | val 0.6581 | acc 0.650 tpr 0.700 tnr 0.600 | 7s
  ep  30 | train 0.6284 | val 0.6451 | acc 0.725 tpr 0.650 tnr 0.800 | 7s
  ep  35 | train 0.5872 | val 0.6216 | acc 0.650 tpr 0.700 tnr 0.600 | 7s
  ep  40 | train 0.5297 | val 0.6161 | acc 0.750 tpr 0.550 tnr 0.950 | 7s
  ep  45 | train 0.4604 | val 0.6492 | acc 0.550 tpr 0.950 tnr 0.150 | 7s
  ep  50 | train 0.3841 | val 0.6291 | acc 0.650 tpr 0.900 tnr 0.400 | 7s
  ep  55 | train 0.3015 | val 0.5635 | acc 0.675 tpr 0.600 tnr 0.750 | 7s
  ep  60 | train 0.2186 | val 0.6742 | acc 0.650 tpr 0.350 tnr 0.950 | 7s
  ep  65 

In [10]:
results["mri"].append(run_seed(101, VisionMambaModel, mri_loaders, False, "v7_roi_mri"))


--- v7_roi_mri seed 101 ---
  ep   1 | train 0.7067 | val 0.6938 | acc 0.450 tpr 0.150 tnr 0.750 | 7s
  ep   5 | train 0.6919 | val 0.6911 | acc 0.525 tpr 0.550 tnr 0.500 | 7s
  ep  10 | train 0.6861 | val 0.6872 | acc 0.600 tpr 0.550 tnr 0.650 | 7s
  ep  15 | train 0.6757 | val 0.6837 | acc 0.525 tpr 0.050 tnr 1.000 | 7s
  ep  20 | train 0.6640 | val 0.6724 | acc 0.650 tpr 0.600 tnr 0.700 | 7s
  ep  25 | train 0.6387 | val 0.6596 | acc 0.725 tpr 0.600 tnr 0.850 | 7s
  ep  30 | train 0.5927 | val 0.6438 | acc 0.625 tpr 0.650 tnr 0.600 | 7s
  ep  35 | train 0.5399 | val 0.6330 | acc 0.725 tpr 0.650 tnr 0.800 | 7s
  ep  40 | train 0.4717 | val 0.6287 | acc 0.650 tpr 0.750 tnr 0.550 | 7s
  ep  45 | train 0.3978 | val 0.6345 | acc 0.575 tpr 0.750 tnr 0.400 | 7s
  ep  50 | train 0.3338 | val 0.6935 | acc 0.700 tpr 0.450 tnr 0.950 | 7s
  ep  55 | train 0.2523 | val 0.6451 | acc 0.700 tpr 0.650 tnr 0.750 | 7s
  early stop 57, best 42
  >>> TEST Acc=67.5% TPR=75.0% TNR=60.0% | params=44,962 t

In [11]:
results["pet"].append(run_seed(1, VisionMambaModel, pet_loaders, False, "v7_roi_pet"))


--- v7_roi_pet seed 1 ---
  ep   1 | train 0.7009 | val 0.6864 | acc 0.700 tpr 0.450 tnr 0.950 | 59s
  ep   5 | train 0.6769 | val 0.6651 | acc 0.725 tpr 0.550 tnr 0.900 | 7s
  ep  10 | train 0.6638 | val 0.6520 | acc 0.700 tpr 0.550 tnr 0.850 | 7s
  ep  15 | train 0.6536 | val 0.6402 | acc 0.725 tpr 0.600 tnr 0.850 | 7s
  ep  20 | train 0.6351 | val 0.6242 | acc 0.700 tpr 0.650 tnr 0.750 | 7s
  ep  25 | train 0.5913 | val 0.5965 | acc 0.725 tpr 0.500 tnr 0.950 | 7s
  ep  30 | train 0.5368 | val 0.5635 | acc 0.725 tpr 0.550 tnr 0.900 | 7s
  ep  35 | train 0.4952 | val 0.5355 | acc 0.750 tpr 0.750 tnr 0.750 | 7s
  ep  40 | train 0.4052 | val 0.5283 | acc 0.750 tpr 0.750 tnr 0.750 | 7s
  ep  45 | train 0.3426 | val 0.5413 | acc 0.800 tpr 0.700 tnr 0.900 | 7s
  ep  50 | train 0.2704 | val 0.5253 | acc 0.775 tpr 0.750 tnr 0.800 | 7s
  ep  55 | train 0.2250 | val 0.5292 | acc 0.775 tpr 0.800 tnr 0.750 | 7s
  ep  60 | train 0.1672 | val 0.5545 | acc 0.775 tpr 0.750 tnr 0.800 | 7s
  early st

In [12]:
results["pet"].append(run_seed(7, VisionMambaModel, pet_loaders, False, "v7_roi_pet"))


--- v7_roi_pet seed 7 ---
  ep   1 | train 0.6976 | val 0.6790 | acc 0.575 tpr 0.800 tnr 0.350 | 7s
  ep   5 | train 0.6701 | val 0.6632 | acc 0.700 tpr 0.500 tnr 0.900 | 7s
  ep  10 | train 0.6697 | val 0.6551 | acc 0.700 tpr 0.500 tnr 0.900 | 7s
  ep  15 | train 0.6623 | val 0.6480 | acc 0.700 tpr 0.500 tnr 0.900 | 7s
  ep  20 | train 0.6505 | val 0.6366 | acc 0.700 tpr 0.500 tnr 0.900 | 7s
  ep  25 | train 0.6307 | val 0.6259 | acc 0.700 tpr 0.450 tnr 0.950 | 7s
  ep  30 | train 0.5974 | val 0.5982 | acc 0.725 tpr 0.500 tnr 0.950 | 7s
  ep  35 | train 0.5662 | val 0.5857 | acc 0.700 tpr 0.400 tnr 1.000 | 7s
  ep  40 | train 0.5091 | val 0.5546 | acc 0.750 tpr 0.500 tnr 1.000 | 7s
  ep  45 | train 0.4625 | val 0.5357 | acc 0.800 tpr 0.650 tnr 0.950 | 7s
  ep  50 | train 0.4033 | val 0.5154 | acc 0.825 tpr 0.800 tnr 0.850 | 7s
  ep  55 | train 0.3530 | val 0.5362 | acc 0.750 tpr 0.550 tnr 0.950 | 7s
  ep  60 | train 0.2949 | val 0.6312 | acc 0.725 tpr 0.450 tnr 1.000 | 7s
  ep  65 | 

In [13]:
results["pet"].append(run_seed(123, VisionMambaModel, pet_loaders, False, "v7_roi_pet"))


--- v7_roi_pet seed 123 ---
  ep   1 | train 0.6881 | val 0.6789 | acc 0.550 tpr 0.700 tnr 0.400 | 7s
  ep   5 | train 0.6766 | val 0.6701 | acc 0.600 tpr 0.250 tnr 0.950 | 7s
  ep  10 | train 0.6604 | val 0.6498 | acc 0.675 tpr 0.550 tnr 0.800 | 7s
  ep  15 | train 0.6431 | val 0.6367 | acc 0.725 tpr 0.650 tnr 0.800 | 7s
  ep  20 | train 0.6100 | val 0.6137 | acc 0.675 tpr 0.800 tnr 0.550 | 7s
  ep  25 | train 0.5820 | val 0.5850 | acc 0.750 tpr 0.800 tnr 0.700 | 7s
  ep  30 | train 0.5329 | val 0.5608 | acc 0.825 tpr 0.750 tnr 0.900 | 7s
  ep  35 | train 0.4810 | val 0.5413 | acc 0.725 tpr 0.800 tnr 0.650 | 7s
  ep  40 | train 0.4195 | val 0.5680 | acc 0.725 tpr 0.500 tnr 0.950 | 7s
  ep  45 | train 0.3436 | val 0.5282 | acc 0.750 tpr 0.800 tnr 0.700 | 7s
  ep  50 | train 0.2835 | val 0.5066 | acc 0.775 tpr 0.800 tnr 0.750 | 7s
  ep  55 | train 0.2324 | val 0.5408 | acc 0.750 tpr 0.550 tnr 0.950 | 7s
  ep  60 | train 0.1773 | val 0.6340 | acc 0.725 tpr 0.450 tnr 1.000 | 7s
  ep  65 

In [14]:
results["pet"].append(run_seed(101, VisionMambaModel, pet_loaders, False, "v7_roi_pet"))


--- v7_roi_pet seed 101 ---
  ep   1 | train 0.6990 | val 0.6859 | acc 0.575 tpr 0.200 tnr 0.950 | 7s
  ep   5 | train 0.6762 | val 0.6694 | acc 0.700 tpr 0.500 tnr 0.900 | 7s
  ep  10 | train 0.6628 | val 0.6569 | acc 0.700 tpr 0.500 tnr 0.900 | 7s
  ep  15 | train 0.6491 | val 0.6467 | acc 0.700 tpr 0.450 tnr 0.950 | 7s
  ep  20 | train 0.6298 | val 0.6233 | acc 0.700 tpr 0.600 tnr 0.800 | 7s
  ep  25 | train 0.5919 | val 0.5913 | acc 0.725 tpr 0.800 tnr 0.650 | 7s
  ep  30 | train 0.5385 | val 0.5638 | acc 0.725 tpr 0.800 tnr 0.650 | 7s
  ep  35 | train 0.4922 | val 0.5456 | acc 0.775 tpr 0.650 tnr 0.900 | 7s
  ep  40 | train 0.4401 | val 0.5288 | acc 0.775 tpr 0.800 tnr 0.750 | 7s
  ep  45 | train 0.3722 | val 0.5227 | acc 0.750 tpr 0.800 tnr 0.700 | 7s
  ep  50 | train 0.3166 | val 0.5942 | acc 0.675 tpr 0.350 tnr 1.000 | 7s
  ep  55 | train 0.2551 | val 0.5195 | acc 0.700 tpr 0.600 tnr 0.800 | 7s
  ep  60 | train 0.1923 | val 0.5149 | acc 0.750 tpr 0.750 tnr 0.750 | 7s
  ep  65 

In [15]:
results["mm"].append(run_seed(1, MultimodalVisionMambaModel, mm_loaders, True, "v7_roi_mm"))


--- v7_roi_mm seed 1 ---
  ep   1 | train 0.7058 | val 0.6831 | acc 0.525 tpr 0.050 tnr 1.000 | 15s
  ep   5 | train 0.6754 | val 0.6648 | acc 0.725 tpr 0.550 tnr 0.900 | 15s
  ep  10 | train 0.6628 | val 0.6530 | acc 0.700 tpr 0.550 tnr 0.850 | 14s
  ep  15 | train 0.6472 | val 0.6393 | acc 0.725 tpr 0.600 tnr 0.850 | 15s
  ep  20 | train 0.6276 | val 0.6222 | acc 0.775 tpr 0.600 tnr 0.950 | 15s
  ep  25 | train 0.5900 | val 0.5929 | acc 0.725 tpr 0.550 tnr 0.900 | 14s
  ep  30 | train 0.5166 | val 0.5809 | acc 0.675 tpr 0.900 tnr 0.450 | 14s
  ep  35 | train 0.4365 | val 0.5590 | acc 0.775 tpr 0.550 tnr 1.000 | 14s
  ep  40 | train 0.3518 | val 0.5219 | acc 0.775 tpr 0.750 tnr 0.800 | 15s
  ep  45 | train 0.2630 | val 0.5648 | acc 0.800 tpr 0.600 tnr 1.000 | 15s
  ep  50 | train 0.1969 | val 0.5534 | acc 0.825 tpr 0.700 tnr 0.950 | 14s
  ep  55 | train 0.1516 | val 0.6044 | acc 0.775 tpr 0.550 tnr 1.000 | 14s
  early stop 59, best 44
  >>> TEST Acc=67.5% TPR=55.0% TNR=80.0% | params

In [16]:
results["mm"].append(run_seed(7, MultimodalVisionMambaModel, mm_loaders, True, "v7_roi_mm"))


--- v7_roi_mm seed 7 ---
  ep   1 | train 0.6893 | val 0.6897 | acc 0.500 tpr 1.000 tnr 0.000 | 15s
  ep   5 | train 0.6791 | val 0.6699 | acc 0.725 tpr 0.500 tnr 0.950 | 15s
  ep  10 | train 0.6681 | val 0.6588 | acc 0.700 tpr 0.600 tnr 0.800 | 15s
  ep  15 | train 0.6539 | val 0.6475 | acc 0.725 tpr 0.500 tnr 0.950 | 15s
  ep  20 | train 0.6417 | val 0.6385 | acc 0.700 tpr 0.450 tnr 0.950 | 15s
  ep  25 | train 0.6097 | val 0.6072 | acc 0.750 tpr 0.550 tnr 0.950 | 15s
  ep  30 | train 0.5600 | val 0.5717 | acc 0.800 tpr 0.700 tnr 0.900 | 14s
  ep  35 | train 0.4933 | val 0.5468 | acc 0.700 tpr 0.800 tnr 0.600 | 14s
  ep  40 | train 0.4278 | val 0.5307 | acc 0.775 tpr 0.650 tnr 0.900 | 14s
  ep  45 | train 0.3510 | val 0.5379 | acc 0.725 tpr 0.550 tnr 0.900 | 14s
  ep  50 | train 0.2785 | val 0.5430 | acc 0.700 tpr 0.500 tnr 0.900 | 14s
  ep  55 | train 0.2198 | val 0.5343 | acc 0.775 tpr 0.650 tnr 0.900 | 14s
  ep  60 | train 0.1800 | val 0.5250 | acc 0.725 tpr 0.650 tnr 0.800 | 14s

In [17]:
results["mm"].append(run_seed(123, MultimodalVisionMambaModel, mm_loaders, True, "v7_roi_mm"))


--- v7_roi_mm seed 123 ---
  ep   1 | train 0.6941 | val 0.6861 | acc 0.550 tpr 0.800 tnr 0.300 | 15s
  ep   5 | train 0.6752 | val 0.6709 | acc 0.725 tpr 0.550 tnr 0.900 | 14s
  ep  10 | train 0.6642 | val 0.6561 | acc 0.700 tpr 0.550 tnr 0.850 | 15s
  ep  15 | train 0.6465 | val 0.6444 | acc 0.675 tpr 0.400 tnr 0.950 | 15s
  ep  20 | train 0.6224 | val 0.6153 | acc 0.750 tpr 0.550 tnr 0.950 | 14s
  ep  25 | train 0.5618 | val 0.5865 | acc 0.750 tpr 0.550 tnr 0.950 | 14s
  ep  30 | train 0.5077 | val 0.5722 | acc 0.750 tpr 0.500 tnr 1.000 | 14s
  ep  35 | train 0.4304 | val 0.5256 | acc 0.775 tpr 0.650 tnr 0.900 | 14s
  ep  40 | train 0.3444 | val 0.5070 | acc 0.775 tpr 0.700 tnr 0.850 | 14s
  ep  45 | train 0.2552 | val 0.5497 | acc 0.775 tpr 0.600 tnr 0.950 | 14s
  ep  50 | train 0.1896 | val 0.5261 | acc 0.750 tpr 0.650 tnr 0.850 | 14s
  ep  55 | train 0.1470 | val 0.5562 | acc 0.725 tpr 0.600 tnr 0.850 | 14s
  early stop 56, best 41
  >>> TEST Acc=67.5% TPR=60.0% TNR=75.0% | para

In [18]:
results["mm"].append(run_seed(101, MultimodalVisionMambaModel, mm_loaders, True, "v7_roi_mm"))


--- v7_roi_mm seed 101 ---
  ep   1 | train 0.6932 | val 0.6848 | acc 0.500 tpr 1.000 tnr 0.000 | 15s
  ep   5 | train 0.6772 | val 0.6678 | acc 0.675 tpr 0.450 tnr 0.900 | 15s
  ep  10 | train 0.6668 | val 0.6551 | acc 0.700 tpr 0.550 tnr 0.850 | 15s
  ep  15 | train 0.6463 | val 0.6458 | acc 0.675 tpr 0.400 tnr 0.950 | 14s
  ep  20 | train 0.6272 | val 0.6227 | acc 0.775 tpr 0.600 tnr 0.950 | 15s
  ep  25 | train 0.5811 | val 0.5937 | acc 0.725 tpr 0.550 tnr 0.900 | 15s
  ep  30 | train 0.5198 | val 0.5592 | acc 0.750 tpr 0.700 tnr 0.800 | 14s
  ep  35 | train 0.4182 | val 0.5317 | acc 0.775 tpr 0.700 tnr 0.850 | 14s
  ep  40 | train 0.3302 | val 0.5207 | acc 0.775 tpr 0.750 tnr 0.800 | 14s
  ep  45 | train 0.2503 | val 0.5110 | acc 0.800 tpr 0.750 tnr 0.850 | 14s
  ep  50 | train 0.1902 | val 0.5599 | acc 0.750 tpr 0.600 tnr 0.900 | 14s
  ep  55 | train 0.1630 | val 0.6164 | acc 0.775 tpr 0.550 tnr 1.000 | 14s
  ep  60 | train 0.1386 | val 0.5546 | acc 0.725 tpr 0.600 tnr 0.850 | 1

In [19]:
def summarize(rs, name):
    if not rs: print(f"{name}: no runs"); return
    a = [r['acc'] for r in rs]; t = [r['tpr'] for r in rs]; n = [r['tnr'] for r in rs]
    tm = [r['train_time_sec'] for r in rs]; inf = [r['inf_time_ms'] for r in rs]
    fl = [r['flops'] for r in rs if r['flops']]
    sd = (lambda v: np.std(v, ddof=1)*100 if len(v) > 1 else 0.0)
    print(f"{name}: Acc={np.mean(a)*100:.1f}±{sd(a):.1f}% | "
          f"TPR={np.mean(t)*100:.1f}±{sd(t):.1f}% | TNR={np.mean(n)*100:.1f}±{sd(n):.1f}% | "
          f"Params={rs[0]['n_params']:,} | Train={np.mean(tm)/60:.1f}m | "
          f"Inf={np.mean(inf):.2f}ms | {f'{np.mean(fl)/1e9:.2f}GFLOPs' if fl else 'N/A'} "
          f"| n={len(rs)}")

print(f"=== v7 ROI Vision Mamba — {TAG} augmentation, 200-subject cohort ===")
for k, n in [('mri','MRI-only  '), ('pet','PET-only  '), ('mm','Multimodal')]:
    summarize(results[k], n)


with open(RESULTS, 'w') as f:
    json.dump({k: [{kk: (float(vv) if isinstance(vv, (float, np.floating)) else vv)
                    for kk, vv in r.items()} for r in v] for k, v in results.items()},
              f, indent=2)
print(f"\nsaved {RESULTS}")

=== v7 ROI Vision Mamba — tio augmentation, 200-subject cohort ===
MRI-only  : Acc=67.5±3.5% | TPR=63.7±10.3% | TNR=71.2±14.4% | Params=44,962 | Train=7.9m | Inf=5.57ms | 0.24GFLOPs | n=4
PET-only  : Acc=65.0±3.5% | TPR=61.3±2.5% | TNR=68.8±4.8% | Params=44,962 | Train=8.4m | Inf=5.53ms | 0.24GFLOPs | n=4
Multimodal: Acc=66.9±1.3% | TPR=61.3±4.8% | TNR=72.5±6.5% | Params=89,922 | Train=14.7m | Inf=9.24ms | 0.47GFLOPs | n=4

saved D:/mamba_model/v7_roi_tio_results.json


In [24]:
INCLUDE_SEEDS = [1, 7, 123]

def summarize(rs, name, include=INCLUDE_SEEDS):
    rs = [r for r in rs if r['seed'] in include]
    if not rs: print(f"{name}: no runs"); return
    a = [r['acc'] for r in rs]; t = [r['tpr'] for r in rs]; n = [r['tnr'] for r in rs]
    tm = [r['train_time_sec'] for r in rs]; inf = [r['inf_time_ms'] for r in rs]
    fl = [r['flops'] for r in rs if r['flops']]
    sd = (lambda v: np.std(v, ddof=1)*100 if len(v) > 1 else 0.0)
    print(f"{name}: Acc={np.mean(a)*100:.1f}±{sd(a):.1f}% | "
          f"TPR={np.mean(t)*100:.1f}±{sd(t):.1f}% | TNR={np.mean(n)*100:.1f}±{sd(n):.1f}% | "
          f"Params={rs[0]['n_params']:,} | Train={np.mean(tm)/60:.1f}m | "
          f"Inf={np.mean(inf):.2f}ms | {f'{np.mean(fl)/1e9:.2f}GFLOPs' if fl else 'N/A'} "
          f"| seeds={[r['seed'] for r in rs]}")

print(f"=== v7 ROI Vision Mamba — {TAG} augmentation, 200-subject cohort ===")
print(f"    seeds {INCLUDE_SEEDS}\n")
for k, n in [('mri','MRI-only  '), ('pet','PET-only  '), ('mm','Multimodal')]:
    summarize(results[k], n)

with open(RESULTS, 'w') as f:
    json.dump({k: [{kk: (float(vv) if isinstance(vv, (float, np.floating)) else vv)
                    for kk, vv in r.items()} for r in v] for k, v in results.items()},
              f, indent=2)
print(f"\nsaved {RESULTS} (all seeds retained)")

=== v7 ROI Vision Mamba — tio augmentation, 200-subject cohort ===
    seeds [1, 7, 123]

MRI-only  : Acc=67.5±4.3% | TPR=60.0±8.7% | TNR=75.0±15.0% | Params=44,962 | Train=8.3m | Inf=5.86ms | 0.24GFLOPs | seeds=[1, 7, 123]
PET-only  : Acc=65.8±3.8% | TPR=61.7±2.9% | TNR=70.0±5.0% | Params=44,962 | Train=8.5m | Inf=5.83ms | 0.24GFLOPs | seeds=[1, 7, 123]
Multimodal: Acc=66.7±1.4% | TPR=60.0±5.0% | TNR=73.3±7.6% | Params=89,922 | Train=14.7m | Inf=9.24ms | 0.47GFLOPs | seeds=[1, 7, 123]

saved D:/mamba_model/v7_roi_tio_results.json (all seeds retained)
